# The pipeline, step by step — what `animate()` does for you

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/main/notebooks/01_pipeline_walkthrough.ipynb)

`animate()` is one call. Underneath it, four modules run in a fixed order, and each
is worth seeing on its own before you trust the video that comes out:

```
aoi.parse          polygon / bbox / file     →  Earth Engine geometry
collection.build   Sentinel-2, cloud-masked  →  + an NDVI band
compositing        one cloud-robust median image per month
render             fetch → colorize → mask no-data → annotate → encode
```

This notebook runs them by hand over the **WNE / Grumsin beech forest** and renders
an annotated Sentinel-2 **NDVI timelapse** (MP4 + GIF). Two areas of interest drive it:

- a **frame** rectangle — the extent that is rendered, and
- a **region** polygon — the area that matters: only scenes with less than
  `region_max_cloud_percent` cloud *over this polygon* are kept, and its outline
  is drawn on every frame.

Makes live Earth Engine calls under your project.

## 1 · Install (Colab only)

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
BRANCH = "main"
URL = f"gee_animation @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"                              # deps + package (first run)
    !pip install -q --force-reinstall --no-deps "{URL}"  # branch moved? force-refresh our code
# pip resolves by version, which doesn't change per commit on a branch --
# --force-reinstall pulls the latest commit. After a code update: Runtime ->
# Restart session, then Run all, so the new module is re-imported.

> **Updated the code on the branch?** After the install cell above, do **Runtime → Restart session**, then **Run all** — otherwise Python keeps the previously-imported (old) module in memory.

## 2 · Authenticate Earth Engine (once)
A separate step — run it a single time. In Colab it opens an `ee.Authenticate()` prompt.
No project of your own yet? [00_setup](00_setup.ipynb#2-·-create-your-own-earth-engine-project)
walks you through creating one (five minutes, free).

In [ ]:
import os
PROJECT = os.environ.get("EE_PROJECT", "")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"

from gee_animation import auth
auth.init(PROJECT)

## 3 · Parameters

Everything about a run lives in one `RunConfig`. `animate()` builds it from its keywords,
a YAML file in `config/` is the same object on disk (notebook 03) — here we write it out
in full. Edit any field and re-run from here down. `viz_min`/`viz_max` + `palette` are
applied identically to every frame so colour is comparable across the animation.

In [ ]:
import urllib.request
from gee_animation.config import RunConfig

AOI_URL = "https://raw.githubusercontent.com/cwinkelmann/GEE_Animation/main/docs/aoi/wne/wne.geojson"
urllib.request.urlretrieve(AOI_URL, "wne.geojson")   # the Grumsin polygon, to the working dir
FRAME = [13.8288, 52.9463, 13.9636, 53.0277]          # [west, south, east, north] — the extent to render

cfg = RunConfig(
    name="wne_ndvi",                     # -> out/wne_ndvi.{mp4,gif}
    project=PROJECT,
    frame_aoi={"bbox": FRAME},           # rectangle: render extent + scene bounds
    region_aoi={"geojson": "wne.geojson"},   # polygon: cloud filter + outline
    region_max_cloud_percent=10,         # keep scenes with < 10% cloud over the region
    start="2022-05-01",                  # inclusive
    end="2022-08-01",                    # exclusive
    sensor="sentinel2",
    cadence="monthly",
    max_cloud_percent=60,                # coarse scene-level pre-filter
    index="ndvi",
    viz_min=-0.2, viz_max=0.9,
    palette=["#a1622f", "#e8d9a0", "#3b7a2a"],   # brown -> tan -> green
    fps=4, scale=20, dimensions=768,
    draw_region=True,                    # draw the region polygon outline on each frame
)
cfg

## 4 · Areas of interest

`aoi.parse` turns either description into an Earth Engine geometry. From here on the
pipeline no longer cares whether you started from a bbox, a GeoJSON or a shapefile.

In [ ]:
from gee_animation import aoi

frame = aoi.parse(cfg.frame_aoi)     # rectangle -> render extent + scene bounds
region = aoi.parse(cfg.region_aoi)   # polygon  -> cloud filter + overlay
print("frame bounds:", frame.bounds().getInfo()["coordinates"])
print("region area (km^2):", round(region.area().getInfo() / 1e6, 2))

## 5 · Build the cloud-masked NDVI collection

Filter Sentinel-2 to the frame, keep scenes with < `region_max_cloud_percent` cloud
*over the region* (computed on the unmasked image), mask clouds with the SCL band, and
add an `NDVI` band — all server-side; nothing has been downloaded yet.

In [ ]:
from gee_animation import collection

coll = collection.build(cfg, frame, region)
n_scenes = coll.size().getInfo()
print(f"{n_scenes} Sentinel-2 scenes pass the frame + region-cloud filter")

## 6 · Monthly median frames

Reduce each month to a cloud-robust median NDVI image. A month with no usable scene is
**skipped, not faked** — the labels printed below are the frames you will get.

In [ ]:
from gee_animation import compositing

frames = compositing.monthly_median(coll, cfg)
print(f"{len(frames)} monthly frames:")
print(", ".join(f.label for f in frames))

## 7 · Preview a single frame

Render just the first frame — including the region outline — to check AOI, palette
and cloud masking before committing to the full animation. This is `render.render`'s
per-frame recipe, unrolled: fetch → colorize → mask no-data → annotate → colorbar → outline.

In [ ]:
from PIL import Image
from IPython.display import display
from gee_animation import render
from gee_animation.imaging import colorize

frame_obj = frames[0]
# The pipeline resolves crs="auto" to the AOI's UTM zone just before fetching
# (render.render does this internally); calling the fetch directly, we do the same.
cfg.crs = render._resolve_crs(cfg, render._aoi_bounds(cfg.frame_aoi))
# render._fetch_thumbnail is the package's internal EE download; used here for a preview.
ndvi_arr, valid = render._fetch_thumbnail(frame_obj.image, cfg, frame)
rgb = colorize(ndvi_arr, cfg.viz_min, cfg.viz_max, cfg.palette)
rgb = render.apply_nodata(rgb, valid)      # cloud / no-data -> neutral grey, never "low NDVI"
rgb = render.annotate(rgb, frame_obj.label)
rgb = render.add_colorbar(rgb, cfg)
if cfg.draw_region:                        # overlay the region polygon outline
    bounds = render._aoi_bounds(cfg.frame_aoi)
    rings = render._region_rings(cfg.region_aoi)
    rgb = render.draw_region(rgb, bounds, rings)
print(f"Preview of {frame_obj.label}:")
display(Image.fromarray(rgb))

## 8 · Render the animation

Run the full pipeline over every frame (region outline drawn automatically when
`cfg.draw_region` is set) and write `out/<name>.mp4` and `out/<name>.gif`.

In [ ]:
paths = render.render(frames, cfg, geometry=frame)
paths

## 9 · Show the result

In [ ]:
from IPython.display import Image as IPyImage

gif = next(p for p in paths if p.suffix == ".gif")
print(f"{gif}")
IPyImage(filename=str(gif))

In [ ]:
from IPython.display import Video

mp4 = next((p for p in paths if p.suffix == ".mp4"), None)
Video(str(mp4), embed=True) if mp4 else "MP4 not produced (ffmpeg unavailable) - see GIF above."

## Make it yours

1. **Your AOI.** Replace the download in §3 with your own polygon (in Colab:
   `from google.colab import files; files.upload()`, then `region_aoi={"geojson": "my_aoi.geojson"}`)
   and set `FRAME` to a bbox a few kilometres wider than it — the picture should always
   show the untouched surroundings, not just your patch.
2. **A stricter cloud filter.** Drop `region_max_cloud_percent` from 10 to 2 and re-run
   §5–§6. Scenes disappear, and with them whole months: the frame list in §6 is where
   you find out, before any pixel is fetched.
3. **The same run in one line.** `animate("wne.geojson", sensor="sentinel2", index="ndvi",
   start="2022-05-01", end="2022-08-01", project=PROJECT)` does everything above — it
   is where the next notebook starts. Compare its frames with yours.

*The seam behind it:* `frames` is a list of `ee.Image`s with a label each. Anything
Earth Engine can do to an image — `reduceRegion` for the region's mean NDVI, an export
to Drive — works on them before `render` ever runs; `render` is only the last step.